# PD-L1 결합체 설계 예제 · v0.2.0

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HisameOgasahara/tmp_protein/blob/main/PDL1_8AOM_T4.ipynb?v=0.2.0)

RosettaCommons 공식 8AOM 예제의 PD-L1 구조와 설계 설정을 사용합니다. T4에서 전체 실행하면 RFD3로 뼈대 8개를 만들고, LigandMPNN으로 뼈대당 서열 2개를 설계한 뒤, RF3로 복합체를 예측합니다.

공식 예제의 뼈대 8개는 T4에서 한 번에 하나씩 생성합니다. 결과 표의 RMSD는 PD-L1을 겹쳤을 때 각 결합체가 처음 설계한 자세에서 얼마나 벗어났는지 보여줍니다. 실행 로그와 구조는 ZIP에 들어 있습니다.

[공식 예제와 입력 출처](https://github.com/RosettaCommons/foundry/blob/0d0b0ba44fab7ca76aa4b625c014ce5964c7d284/models/rfd3/docs/tutorials/binder_design_tutorial.md)


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
import subprocess
import sys

checkout = Path(mkdtemp(prefix="gfp_main_", dir="/content")) / "repo"
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--single-branch", "--branch", "main",
     "https://github.com/HisameOgasahara/tmp_protein.git", str(checkout)],
    capture_output=True, text=True,
)
if clone.returncode:
    print(clone.stderr)
    clone.check_returncode()
revision = subprocess.check_output(["git", "-C", str(checkout), "rev-parse", "HEAD"], text=True).strip()
print(f"main: {revision[:12]}")


In [ ]:
import pandas as pd
from IPython.display import display

log = checkout.parent / "pdl1_workflow.log"
with log.open("w", encoding="utf-8") as handle:
    with subprocess.Popen(
        [sys.executable, "-u", str(checkout / "pdl1_pipeline/workflow.py"), revision],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8",
    ) as process:
        for line in process.stdout:
            handle.write(line)
            handle.flush()
            print(line, end="", flush=True)
            if line.strip().endswith(".zip"):
                result_zip = Path(line.strip())
        if process.wait():
            raise RuntimeError(f"Workflow failed. Full log: {log}")

display(pd.read_csv(result_zip.with_suffix("") / "complexes.csv"))


In [ ]:
from google.colab import files
files.download(str(result_zip))
